In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Generar y Explorar Datos de Transacciones

In [ ]:
# Generar dataset sintético de transacciones de supermercado
np.random.seed(42)

# Productos disponibles
productos = [
    'Pan', 'Leche', 'Huevos', 'Mantequilla', 'Queso',
    'Yogurt', 'Cereal', 'Café', 'Té', 'Azúcar',
    'Arroz', 'Pasta', 'Tomate', 'Lechuga', 'Cebolla',
    'Manzanas', 'Plátanos', 'Naranjas', 'Pollo', 'Carne'
]

# Definir combinaciones frecuentes (para simular patrones reales)
combinaciones_comunes = [
    ['Pan', 'Leche', 'Huevos'],
    ['Pan', 'Mantequilla'],
    ['Leche', 'Cereal'],
    ['Café', 'Azúcar'],
    ['Té', 'Azúcar'],
    ['Pasta', 'Tomate'],
    ['Arroz', 'Pollo'],
    ['Manzanas', 'Plátanos', 'Naranjas'],
    ['Lechuga', 'Tomate', 'Cebolla'],
    ['Queso', 'Yogurt', 'Leche']
]

# Generar transacciones
transacciones = []
n_transacciones = 200

for i in range(n_transacciones):
    # Decidir si usar una combinación común (60% de probabilidad)
    if np.random.random() < 0.6:
        # Elegir una combinación común
        transaccion = combinaciones_comunes[np.random.randint(0, len(combinaciones_comunes))].copy()
        # Añadir algunos productos aleatorios adicionales
        n_adicionales = np.random.randint(0, 4)
        productos_adicionales = np.random.choice(productos, n_adicionales, replace=False)
        transaccion.extend(productos_adicionales)
    else:
        # Transacción completamente aleatoria
        n_items = np.random.randint(2, 7)
        transaccion = list(np.random.choice(productos, n_items, replace=False))
    
    # Eliminar duplicados
    transaccion = list(set(transaccion))
    transacciones.append(transaccion)

print(f"Total de transacciones generadas: {len(transacciones)}")
print(f"\nEjemplos de transacciones:")
for i, trans in enumerate(transacciones[:10], 1):
    print(f"Transacción {i}: {trans}")

In [ ]:
# Estadísticas básicas
longitudes = [len(trans) for trans in transacciones]

print("Estadísticas de las transacciones:")
print(f"Longitud promedio: {np.mean(longitudes):.2f} items")
print(f"Longitud mínima: {np.min(longitudes)} items")
print(f"Longitud máxima: {np.max(longitudes)} items")
print(f"Total de productos únicos: {len(productos)}")

In [ ]:
# Visualizar distribución de longitudes de transacciones
plt.figure(figsize=(10, 5))
plt.hist(longitudes, bins=range(1, max(longitudes)+2), edgecolor='black', alpha=0.7)
plt.xlabel('Número de Items por Transacción')
plt.ylabel('Frecuencia')
plt.title('Distribución de Longitud de Transacciones')
plt.xticks(range(1, max(longitudes)+1))
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Frecuencia de productos individuales
from collections import Counter

todos_productos = [item for trans in transacciones for item in trans]
frecuencia_productos = Counter(todos_productos)

# Crear DataFrame
df_frecuencia = pd.DataFrame.from_dict(frecuencia_productos, orient='index', columns=['Frecuencia'])
df_frecuencia = df_frecuencia.sort_values('Frecuencia', ascending=False)

print("Top 10 productos más comprados:")
print(df_frecuencia.head(10))

# Visualización
plt.figure(figsize=(12, 6))
df_frecuencia.head(15).plot(kind='barh', legend=False, color='steelblue')
plt.xlabel('Frecuencia')
plt.title('Top 15 Productos Más Comprados')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Preparar los Datos para Apriori

In [ ]:
# Convertir las transacciones a formato binario (One-Hot Encoding)
te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)
df_transacciones = pd.DataFrame(te_array, columns=te.columns_)

print("Dataset en formato binario:")
print(f"Shape: {df_transacciones.shape}")
print(f"\nPrimeras filas:")
print(df_transacciones.head(10))

## 3. Aplicar el Algoritmo Apriori

In [ ]:
# Encontrar conjuntos de items frecuentes
# min_support: frecuencia mínima para considerar un conjunto de items
min_support = 0.05  # 5% de las transacciones

print(f"Buscando conjuntos frecuentes con soporte mínimo de {min_support*100}%...\n")

frequent_itemsets = apriori(df_transacciones, min_support=min_support, use_colnames=True)

print(f"Total de conjuntos frecuentes encontrados: {len(frequent_itemsets)}")
print(f"\nConjuntos frecuentes:")
print(frequent_itemsets.sort_values('support', ascending=False).head(20))

In [ ]:
# Añadir columna con la longitud del itemset
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("Distribución por longitud de itemset:")
print(frequent_itemsets['length'].value_counts().sort_index())

# Visualización
plt.figure(figsize=(10, 5))
frequent_itemsets['length'].value_counts().sort_index().plot(kind='bar', 
                                                               color='coral', 
                                                               edgecolor='black')
plt.xlabel('Longitud del Itemset')
plt.ylabel('Cantidad')
plt.title('Distribución de Itemsets Frecuentes por Longitud')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Generar Reglas de Asociación

In [ ]:
# Generar reglas de asociación
# metric: métrica para filtrar reglas (confidence, lift, support)
# min_threshold: umbral mínimo para la métrica

rules = association_rules(frequent_itemsets, 
                         metric="confidence", 
                         min_threshold=0.3,
                         num_itemsets=len(frequent_itemsets))

print(f"Total de reglas de asociación generadas: {len(rules)}")
print(f"\nPrimeras reglas:")
print(rules.head())

In [ ]:
# Ordenar reglas por diferentes métricas
print("\n=== TOP 10 REGLAS POR CONFIANZA ===")
top_confidence = rules.sort_values('confidence', ascending=False).head(10)
for idx, row in top_confidence.iterrows():
    print(f"{list(row['antecedents'])} → {list(row['consequents'])}")
    print(f"  Soporte: {row['support']:.3f} | Confianza: {row['confidence']:.3f} | Lift: {row['lift']:.3f}\n")

In [ ]:
print("\n=== TOP 10 REGLAS POR LIFT ===")
top_lift = rules.sort_values('lift', ascending=False).head(10)
for idx, row in top_lift.iterrows():
    print(f"{list(row['antecedents'])} → {list(row['consequents'])}")
    print(f"  Soporte: {row['support']:.3f} | Confianza: {row['confidence']:.3f} | Lift: {row['lift']:.3f}\n")

## 5. Visualizar Reglas de Asociación

In [ ]:
# Scatter plot: Soporte vs Confianza (coloreado por Lift)
plt.figure(figsize=(12, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], 
                     c=rules['lift'], cmap='viridis', 
                     s=100, alpha=0.6, edgecolors='k')
plt.xlabel('Soporte')
plt.ylabel('Confianza')
plt.title('Reglas de Asociación: Soporte vs Confianza (color = Lift)')
plt.colorbar(scatter, label='Lift')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar distribuciones de métricas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rules['support'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Soporte')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Soporte')

axes[1].hist(rules['confidence'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Confianza')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Confianza')

axes[2].hist(rules['lift'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_xlabel('Lift')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Lift')

plt.tight_layout()
plt.show()

## 6. Filtrar Reglas Más Relevantes

In [ ]:
# Filtrar reglas con alta confianza y lift
reglas_fuertes = rules[
    (rules['confidence'] >= 0.5) & 
    (rules['lift'] >= 1.5)
].sort_values('lift', ascending=False)

print(f"Reglas fuertes (Confianza ≥ 0.5 y Lift ≥ 1.5): {len(reglas_fuertes)}")
print("\nReglas más fuertes encontradas:\n")

if len(reglas_fuertes) > 0:
    for idx, row in reglas_fuertes.head(15).iterrows():
        ant = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        print(f"Si compra [{ant}] → entonces [{cons}]")
        print(f"  📊 Soporte: {row['support']:.1%} | Confianza: {row['confidence']:.1%} | Lift: {row['lift']:.2f}")
        print()
else:
    print("No se encontraron reglas que cumplan los criterios.")
    print("\nMejores reglas disponibles:")
    for idx, row in rules.sort_values('lift', ascending=False).head(10).iterrows():
        ant = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        print(f"Si compra [{ant}] → entonces [{cons}]")
        print(f"  📊 Soporte: {row['support']:.1%} | Confianza: {row['confidence']:.1%} | Lift: {row['lift']:.2f}")
        print()

## 7. Análisis de Productos Específicos

In [ ]:
# Buscar reglas que involucren un producto específico
producto_interes = 'Leche'

# Reglas donde el producto aparece en el antecedente
reglas_con_producto = rules[
    rules['antecedents'].apply(lambda x: producto_interes in x)
].sort_values('confidence', ascending=False)

print(f"Reglas donde '{producto_interes}' aparece en el antecedente (¿qué compran con {producto_interes}?):")
print(f"Total: {len(reglas_con_producto)} reglas\n")

if len(reglas_con_producto) > 0:
    for idx, row in reglas_con_producto.head(10).iterrows():
        cons = ', '.join(list(row['consequents']))
        print(f"[{producto_interes}] → [{cons}]")
        print(f"  Confianza: {row['confidence']:.1%} | Lift: {row['lift']:.2f}\n")
else:
    print(f"No se encontraron reglas con {producto_interes} en el antecedente.")

In [ ]:
# Reglas donde el producto aparece en el consecuente
reglas_hacia_producto = rules[
    rules['consequents'].apply(lambda x: producto_interes in x)
].sort_values('confidence', ascending=False)

print(f"Reglas donde '{producto_interes}' aparece en el consecuente (¿qué lleva a comprar {producto_interes}?):")
print(f"Total: {len(reglas_hacia_producto)} reglas\n")

if len(reglas_hacia_producto) > 0:
    for idx, row in reglas_hacia_producto.head(10).iterrows():
        ant = ', '.join(list(row['antecedents']))
        print(f"[{ant}] → [{producto_interes}]")
        print(f"  Confianza: {row['confidence']:.1%} | Lift: {row['lift']:.2f}\n")
else:
    print(f"No se encontraron reglas con {producto_interes} en el consecuente.")

## 8. Resumen y Recomendaciones

In [ ]:
# Crear resumen de hallazgos
print("="*60)
print("RESUMEN DEL ANÁLISIS DE CANASTA DE MERCADO")
print("="*60)

print(f"\n📊 Estadísticas Generales:")
print(f"  • Total de transacciones analizadas: {len(transacciones)}")
print(f"  • Productos únicos: {len(productos)}")
print(f"  • Items promedio por transacción: {np.mean(longitudes):.2f}")
print(f"  • Conjuntos frecuentes encontrados: {len(frequent_itemsets)}")
print(f"  • Reglas de asociación generadas: {len(rules)}")

print(f"\n🎯 Métricas de Reglas:")
print(f"  • Soporte promedio: {rules['support'].mean():.3f}")
print(f"  • Confianza promedio: {rules['confidence'].mean():.3f}")
print(f"  • Lift promedio: {rules['lift'].mean():.3f}")

print(f"\n🔝 Top 3 Productos Más Comprados:")
for i, (producto, freq) in enumerate(df_frecuencia.head(3).iterrows(), 1):
    porcentaje = (freq['Frecuencia'] / len(transacciones)) * 100
    print(f"  {i}. {producto}: {freq['Frecuencia']} veces ({porcentaje:.1f}% de transacciones)")

print(f"\n💡 Recomendaciones de Negocio:")
print("  1. Colocar productos frecuentemente comprados juntos cerca en la tienda")
print("  2. Crear promociones de paquetes basadas en reglas de alta confianza")
print("  3. Usar reglas para recomendaciones de productos en línea")
print("  4. Optimizar inventario basándose en patrones de compra")
print("="*60)

## Conclusiones

- **Apriori** es un algoritmo eficiente para descubrir patrones de compra en grandes volúmenes de transacciones
- Las **reglas de asociación** revelan qué productos se compran juntos frecuentemente
- **Soporte** indica qué tan frecuente es la combinación en el dataset
- **Confianza** muestra la probabilidad condicional de compra
- **Lift > 1** indica que los productos se compran juntos más frecuentemente que por azar
- Estas reglas son valiosas para:
  - Colocación estratégica de productos
  - Promociones cruzadas
  - Sistemas de recomendación
  - Optimización de inventario